# Automated Construction of Fe | MgO Interfaces

## Objective

Systematically generate and evaluate possible Fe | MgO interfaces using computational methods.

The workflow includes:
1. Enumeration of crystallographic surfaces
2. Lattice matching using ZSL algorithm
3. Interface construction
4. Geometric optimization
5. Chemical and strain-based scoring
6. Selection of optimal interface

In [1]:
from dash import Dash
import crystal_toolkit.components as ctc
from pymatgen.core import Structure
import crystal_toolkit
import numpy as np
from multiprocessing.dummy import Pool
from pymatgen.ext.matproj import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core.surface import get_symmetrically_distinct_miller_indices
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

/home/akanksha/Siesta_5/miniconda3/envs/matsci-ct311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Physical Background

An ideal interface should:

- Minimize lattice mismatch → reduces strain
- Maximize favorable bonding (e.g., O–Mg)
- Avoid atomic overlap (short distances)
- Maintain crystallographic coherence

The total interface energy can be approximated as:
E_total ≈ E_strain + E_interface + E_chemical

## Load Bulk Structures

We obtain Fe (bcc) and MgO (rocksalt) structures from Materials Project.

Structures are converted to conventional standard cells to ensure:
- consistent symmetry
- correct lattice orientation

In [2]:
API_KEY = "pPxBVAcW8uMoR8udpokQXBGFSpiEX39d" 

with MPRester(API_KEY) as mpr:
    fe_bulk = mpr.get_structure_by_material_id("mp-13")
    mgo_bulk = mpr.get_structure_by_material_id("mp-1265")

fe_bulk = SpacegroupAnalyzer(fe_bulk).get_conventional_standard_structure()
mgo_bulk = SpacegroupAnalyzer(mgo_bulk).get_conventional_standard_structure()

## Surface Enumeration

We restrict Miller indices to ≤ 1 to:
- reduce computational cost
- focus on physically relevant low-energy surfaces

Examples:
- Fe (110), (100), (111)
- MgO (100), (110), (111)

In [3]:
#Enumerate Miller Planes - limiting Miller index to 2.

max_index = 1

fe_planes = get_symmetrically_distinct_miller_indices(fe_bulk, max_index)
mgo_planes = get_symmetrically_distinct_miller_indices(mgo_bulk, max_index)

print("Fe planes:", fe_planes)
print("MgO planes:", mgo_planes)

Fe planes: [(1, 1, 1), (1, 1, 0), (1, 0, 0)]
MgO planes: [(1, 1, 1), (1, 1, 0), (1, 0, 0)]


## Lattice Matching using ZSL Algorithm

The Zur–McGill (ZSL) algorithm finds supercells of two lattices such that:

- lattice vectors match within tolerance
- mismatch is minimized

This enables coherent interface construction.

Parameters used:
- max_area → limits supercell size
- length_tol → controls mismatch tolerance
- angle_tol → controls angular deviation

In [3]:
from pymatgen.analysis.interfaces.zsl import ZSLGenerator, ZSLMatch
from pymatgen.analysis.interfaces.substrate_analyzer import SubstrateAnalyzer, SubstrateMatch

zsl = ZSLGenerator(
    max_area_ratio_tol = 0.3,
    max_area = 400,
    max_length_tol = 0.1,
    max_angle_tol = 0.1
)


## Interface Construction

For each (Fe plane, MgO plane) pair:

- CoherentInterfaceBuilder generates:
- possible terminations
- lattice-matched interfaces

Each interface differs in:
- stacking
- termination
- supercell size

## Interface Properties

For each interface, we compute:

### 1. Interface Area
A = |a × b|

This represents:
- surface size
- affects interface energy

### 2. Strain (von Mises strain)

Represents lattice distortion due to matching.
Lower strain → more stable interface

In [3]:
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

cib_objects = []

for fe_plane in fe_planes:
    for mgo_plane in mgo_planes:
        
        try:
            cib = CoherentInterfaceBuilder(
                substrate_structure=fe_bulk,
                film_structure=mgo_bulk,
                substrate_miller=fe_plane,
                film_miller=mgo_plane,
                zslgen=zsl
            )

            cib_objects.append({
                "fe_plane": fe_plane,
                "mgo_plane": mgo_plane,
                "cib": cib
            })

        except:
            continue

print("Total CIB objects:", len(cib_objects))

NameError: name 'fe_planes' is not defined

In [6]:
terminations = cib.terminations
print("terminations:", len(terminations))
for t in terminations:
    print(t)

terminations: 1
('MgO_P4/mmm_2', 'Fe_P4/mmm_1')


In [7]:
termination_info = []

for item in cib_objects:
    
    cib = item["cib"]
    n_terminations = len(cib.terminations)

    termination_info.append({
        "fe_plane": item["fe_plane"],
        "mgo_plane": item["mgo_plane"],
        "terminations": n_terminations
    })

termination_info

[{'fe_plane': (1, 1, 1), 'mgo_plane': (1, 1, 1), 'terminations': 1},
 {'fe_plane': (1, 1, 1), 'mgo_plane': (1, 1, 0), 'terminations': 1},
 {'fe_plane': (1, 1, 1), 'mgo_plane': (1, 0, 0), 'terminations': 1},
 {'fe_plane': (1, 1, 0), 'mgo_plane': (1, 1, 1), 'terminations': 1},
 {'fe_plane': (1, 1, 0), 'mgo_plane': (1, 1, 0), 'terminations': 1},
 {'fe_plane': (1, 1, 0), 'mgo_plane': (1, 0, 0), 'terminations': 1},
 {'fe_plane': (1, 0, 0), 'mgo_plane': (1, 1, 1), 'terminations': 1},
 {'fe_plane': (1, 0, 0), 'mgo_plane': (1, 1, 0), 'terminations': 1},
 {'fe_plane': (1, 0, 0), 'mgo_plane': (1, 0, 0), 'terminations': 1}]

In [8]:
matches = list(cib.zsl_matches)
print(len(matches))

75405


In [9]:
match_info = []

for item in cib_objects:

    cib = item["cib"]

    matches = cib.zsl_matches   # ← correct attribute

    match_info.append({
        "fe_plane": item["fe_plane"],
        "mgo_plane": item["mgo_plane"],
        "matches": len(matches)
    })

match_info

[{'fe_plane': (1, 1, 1), 'mgo_plane': (1, 1, 1), 'matches': 34348},
 {'fe_plane': (1, 1, 1), 'mgo_plane': (1, 1, 0), 'matches': 12958},
 {'fe_plane': (1, 1, 1), 'mgo_plane': (1, 0, 0), 'matches': 23348},
 {'fe_plane': (1, 1, 0), 'mgo_plane': (1, 1, 1), 'matches': 211014},
 {'fe_plane': (1, 1, 0), 'mgo_plane': (1, 1, 0), 'matches': 76777},
 {'fe_plane': (1, 1, 0), 'mgo_plane': (1, 0, 0), 'matches': 150929},
 {'fe_plane': (1, 0, 0), 'mgo_plane': (1, 1, 1), 'matches': 94239},
 {'fe_plane': (1, 0, 0), 'mgo_plane': (1, 1, 0), 'matches': 38611},
 {'fe_plane': (1, 0, 0), 'mgo_plane': (1, 0, 0), 'matches': 75405}]

In [10]:
summary = []

for item in cib_objects:

    cib = item["cib"]

    n_terminations = len(cib.terminations)
    n_matches = len(cib.zsl_matches)

    summary.append({
        "fe_plane": item["fe_plane"],
        "mgo_plane": item["mgo_plane"],
        "terminations": n_terminations,
        "matches": n_matches,
        "total_interfaces": n_terminations * n_matches
    })

summary

[{'fe_plane': (1, 1, 1),
  'mgo_plane': (1, 1, 1),
  'terminations': 1,
  'matches': 34348,
  'total_interfaces': 34348},
 {'fe_plane': (1, 1, 1),
  'mgo_plane': (1, 1, 0),
  'terminations': 1,
  'matches': 12958,
  'total_interfaces': 12958},
 {'fe_plane': (1, 1, 1),
  'mgo_plane': (1, 0, 0),
  'terminations': 1,
  'matches': 23348,
  'total_interfaces': 23348},
 {'fe_plane': (1, 1, 0),
  'mgo_plane': (1, 1, 1),
  'terminations': 1,
  'matches': 211014,
  'total_interfaces': 211014},
 {'fe_plane': (1, 1, 0),
  'mgo_plane': (1, 1, 0),
  'terminations': 1,
  'matches': 76777,
  'total_interfaces': 76777},
 {'fe_plane': (1, 1, 0),
  'mgo_plane': (1, 0, 0),
  'terminations': 1,
  'matches': 150929,
  'total_interfaces': 150929},
 {'fe_plane': (1, 0, 0),
  'mgo_plane': (1, 1, 1),
  'terminations': 1,
  'matches': 94239,
  'total_interfaces': 94239},
 {'fe_plane': (1, 0, 0),
  'mgo_plane': (1, 1, 0),
  'terminations': 1,
  'matches': 38611,
  'total_interfaces': 38611},
 {'fe_plane': (1, 0,

## Interface Enumeration
All terminations and ZSL matches are used to generate interfaces.

In [ ]:
all_interfaces = []

for item in cib_objects:
    cib = item["cib"]
    for term in cib.terminations:
        interfaces = list(
            cib.get_interfaces(
                termination=term,
                gap=1.75,
                film_thickness=4,
                substrate_thickness=4
            )
        )
        all_interfaces.extend(interfaces)

print("Total interfaces generated:", len(all_interfaces))

In [ ]:
interfaces_data = []

for interface in interfaces:
    print(interface.interface_properties)

In [ ]:
interfaces_least_strain = min(all_interfaces,
           key=lambda x: x.interface_properties['von_mises_strain'])

print("Strain:", interfaces_least_strain.interface_properties['von_mises_strain']*100, "%")

## Interface Property Calculation
We compute interface area and extract strain.

In [ ]:
for interface in all_interfaces:

    print("Fe transformation matrix:")
    print(interface.interface_properties["film_transformation"])

    print("MgO transformation matrix:")
    print(interface.interface_properties["substrate_transformation"])
    print("--------------------------")

In [ ]:
import numpy as np

for interface in all_interfaces:
    
    vecs = interface.interface_properties["substrate_sl_vectors"]
    
    a = vecs[0]
    b = vecs[1]
    
    area = np.linalg.norm(np.cross(a, b))
    
    interface.interface_properties["area"] = area

for interface in all_interfaces:
    print(
        interface.interface_properties["von_mises_strain"],
        interface.interface_properties["area"]
    )

## Chemical Scoring Function

Chemical scoring based on:

### Rules:

1. Favor O-Mg bonding  
   - mimics ionic bonding in MgO  

2. Penalize same-species interactions  
   - avoids unphysical contacts (O-O, Zn-Zn, Mg-Mg)

3. Consider only short-range interactions (d < 2.5 Å)

This approximates interfacial bonding energy.

## Geometric Optimization

Two optimizations are performed:

### 1. Interlayer Distance Optimization
- Shift film along z-direction
- Find distance minimizing atomic overlap

### 2. In-plane Translation
- Shift film in x-y plane
- Explore registry configurations

This mimics atomic relaxation at interfaces.

## Final Scoring Function

The total score is defined as:

Score = w1 × strain + w2 × area + w3 × chemical_score

Where:
- strain → penalizes lattice distortion
- area → penalizes large supercells
- chemical_score → accounts for bonding

Lower score → better interface

In [ ]:
def score(interface):

    strain = interface.interface_properties["von_mises_strain"]
    area = interface.interface_properties["area"]

    return strain + area / 50


final_interface = min(all_interfaces, key=score)
final_interface

In [28]:
print("Fe transformation matrix:")
print(final_interface.interface_properties["substrate_transformation"])

print("MgO transformation matrix:")
print(final_interface.interface_properties["film_transformation"])

Fe transformation matrix:
[[1. 0.]
 [0. 1.]]
MgO transformation matrix:
[[1. 0.]
 [0. 1.]]


In [29]:
import numpy as np

f = final_interface.interface_properties["substrate_sl_vectors"][0]
s = final_interface.interface_properties["film_sl_vectors"][0]

angle = np.degrees(
        np.arccos(
        np.dot(f,s) / (np.linalg.norm(f)*np.linalg.norm(s))
    )
)

print("Rotation angle:", angle)

Rotation angle: 135.0


In [30]:
print("Fe interface lattice vectors:")
print(final_interface.interface_properties["substrate_sl_vectors"])

print("MgO interface lattice vectors:")
print(final_interface.interface_properties["film_sl_vectors"])

Fe interface lattice vectors:
[[4.60411223e-16 2.86303550e+00 1.75310363e-16]
 [0.00000000e+00 0.00000000e+00 2.86303550e+00]]
MgO interface lattice vectors:
[[-3.37223544e-16 -2.09700139e+00 -2.09700139e+00]
 [-3.37223544e-16 -2.09700139e+00  2.09700139e+00]]


In [31]:
strain = final_interface.interface_properties["strain"]
print(strain)

[[ 0.00000000e+00 -5.46587076e-18  5.58466691e-33]
 [-5.46587076e-18 -3.39891411e-02  1.30186399e-19]
 [ 5.58466691e-33  1.30186399e-19 -3.39891411e-02]]


In [32]:
import numpy as np

a_fe = fe_bulk.lattice.a
a_mgo = mgo_bulk.lattice.a

print("Fe lattice:", a_fe)
print("MgO lattice:", a_mgo)

print("√2 * a_Fe =", np.sqrt(2)*a_fe)

mismatch = abs(np.sqrt(2)*a_fe - a_mgo)/a_mgo * 100
print("Mismatch:", mismatch, "%")

Fe lattice: 2.863035498949916
MgO lattice: 4.1940027853522395
√2 * a_Fe = 4.048943632170593
Mismatch: 3.458728107865664 %


## Limitations

- No electronic structure calculations (DFT not included)
- Chemical score is heuristic
- No relaxation using force minimization
- Does not include temperature effects

In [33]:
print(final_interface.interface_properties["film_transformation"])
print(final_interface.interface_properties["substrate_transformation"])

[[1. 0.]
 [0. 1.]]
[[1. 0.]
 [0. 1.]]


In [4]:
API_KEY = "pPxBVAcW8uMoR8udpokQXBGFSpiEX39d" 

with MPRester(API_KEY) as mpr:
    fe_bulk = mpr.get_structure_by_material_id("mp-13")
    zno_bulk = mpr.get_structure_by_material_id("mp-2133")

fe_bulk = SpacegroupAnalyzer(fe_bulk).get_conventional_standard_structure()
zno_bulk = SpacegroupAnalyzer(zno_bulk).get_conventional_standard_structure()

In [5]:
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder

cib = CoherentInterfaceBuilder(
                substrate_structure=zno_bulk,
                film_structure=fe_bulk,
                substrate_miller=(0,0,1),
                film_miller=(1,1,1),
                zslgen=zsl
            )

term = cib.terminations
for t in term:
    print(t)

('Fe_R-3m_1', 'Zn_P6/mmm_1')
('Fe_R-3m_1', 'O2_P6/mmm_1')


In [25]:
term1 = cib.terminations[1]
interface = list(cib.get_interfaces(term1))
interface

[Structure Summary
 Lattice
     abc : 6.474702046908749 6.474702046908748 27.417340153669674
  angles : 90.0 90.0 60.00000000000001
  volume : 995.3951664604427
       A : np.float64(6.474702046908749) np.float64(0.0) np.float64(3.9646115685898073e-16)
       B : np.float64(3.237351023454373) np.float64(5.6072564545580805) np.float64(3.9646115685898073e-16)
       C : np.float64(0.0) np.float64(0.0) np.float64(27.417340153669674)
     pbc : True True True
 PeriodicSite: Zn (3.237, 1.869, 10.13) [0.3333, 0.3333, 0.3694]
 PeriodicSite: Zn (4.856, 4.673, 10.13) [0.3333, 0.8333, 0.3694]
 PeriodicSite: Zn (6.475, 1.869, 10.13) [0.8333, 0.3333, 0.3694]
 PeriodicSite: Zn (8.093, 4.673, 10.13) [0.8333, 0.8333, 0.3694]
 PeriodicSite: Zn (1.619, 0.9345, 12.74) [0.1667, 0.1667, 0.4647]
 PeriodicSite: Zn (3.237, 3.738, 12.74) [0.1667, 0.6667, 0.4647]
 PeriodicSite: Zn (4.856, 0.9345, 12.74) [0.6667, 0.1667, 0.4647]
 PeriodicSite: Zn (6.475, 3.738, 12.74) [0.6667, 0.6667, 0.4647]
 PeriodicSite: Fe

In [ ]:
from pymatgen.analysis.local_env import CrystalNN
import numpy as np

# Generate all interfaces from all terminations

all_interfaces = []
for term in cib.terminations:
    interfaces = list(cib.get_interfaces(
            termination=term,
            gap=2.0,
            film_thickness=4,
            substrate_thickness=4))

    for interface in interfaces:
        interface.interface_properties["termination"] = term
        all_interfaces.append(interface)

print("Total interfaces:", len(all_interfaces))